# CASE STUDY:  
## Computation of PI, area of circle & sphere using Montecarlo method:  
This section presents the case study, in which PI, area of circle & volume of sphere has been computed. The program is written in Python. This case study includes the use of following concepts;

1. File IO.  
2. NumPy array.  
3. Random Number Generator.  
4. Montecarlo method.  
5. MPI for parallelization of code.  
6. Collective Communication operations (Broadcast and reduce).
   
Montecarlo method is a statistical technique used to estimate the numerical quantities using random sampling. In this case study, Montecarlo method is used to estimate following quantities;  
1. The value of Pi.  
2. The area of Circle.  
3. The volume of Sphere.
 
Montecarlo method works by generating random points inside a known geometric region and estimate the quantity by determining the fraction of points that fall inside the target shape.

     Probability = (Points_inside) / (Total_points)
   
For this case study, it is considered that circle of radius R is inscribed in a square of area equal 1. The random points are generated inside the square, the probability that random points fall inside the circle is given by;

$$ \frac{\text{Points Inside Circle}}
{\text{Total Points}}
=\frac{\text{Area of circle}}
{\text{Area of square}}
=\frac{\pi (0.5)^2}{1}
=\frac{\pi}{4}
$$

Where Pi can be estimated as;

$$
\pi_{estimate}
=
4 \left(
\frac{\text{Points Inside Circle}}
{\text{Total Points}}
\right)
$$

For area of circle;

$$
\text{Area of Circle}
=
\pi_{estimate} r^2
$$

In the same way this can be done for sphere, volume of sphere is given by;

$$
V
=
\frac{4}{3}\pi r^3
$$

Where sphere is enclosed inside a cube of volume equals to 1.

$$
\frac{\text{Points Inside Target Shape}}
{\text{Total Points}}
=
\frac{\text{Volume of Sphere}}
{\text{Volume of Cube}}
=
\frac{\pi}{6}
$$
                       
Where Pi can be estimated as;

$$
\pi_{estimate}
=
6
\left(
\frac{\text{Points Inside Sphere}}
{\text{Total Points}}
\right)
$$

Volume of sphere can be computed as;

$$
V
=
\frac{4}{3}\pi_{estimate} r^3
$$

To start, program reads the input file for initial data:  
total_points=1000000  
dimensions=2  
radius=0.5  
convergence=1.0e-6  

Once the total number of points are read, the root process, usually process 0 send this data to all the process in communicator using the broadcast operation. The idea is to divide total number of points in N process, so that each process uses Montecarlo method to see how many points lie inside the circle. The program is written to handle the situation where number point distributed among process may not be equal.

The point is considered to be inside circle if;    
$$
(x-0.5)^2+(y-0.5)^2 \le 0.25
$$

Each process computes the number of points lie inside circle and send this data to root using Reduce which is collective communication mpi operation. It sums up all the data and final result is received in root process. Root process then estimates Pi, if the estimated Pi converge to a threshold value, the value is considered and area of circle is then computed, otherwise root process increments total number of points and broadcast it again all the process. Then the process repeats itself until the value of estimated Pi is converged.

The same process is followed for Pi estimate and volume of sphere, with only difference that dimension variable is now equals 3.

At the end of process, the results are written in the output file along with the computation time.

The program is run with processes 2, 4, 8 ,16, 20.

The effect of strong scaling is also presented in this study.

Strong Scaling refers to how execution time of fixed size problem decreases as the number of processors increases.

Speedup is given by;  

$$
S(p)
=
\frac{T(1)}
{T(p)}
$$

$$
\text{Efficiency}(p)_strong
=
\frac{S(p)}
{p}
$$


The python code of program is reported below;

In [ ]:
"""
Distributed Monte Carlo Simulation for Area (2D) and Volume (3D) calculations.
This script demonstrates the use of MPI Bcast and Reduce for compute-bound tasks.
"""

import os
import numpy as np
from mpi4py import MPI
import math

FILE_CONFIG = "/home/zia__/jupyter/mc_config.txt"
FILE_OUTPUT = "/home/zia__/jupyter/mc_results_mpi.txt"

def load_or_create_config(filename):
    """
    Reads parameters from a text file. Creates it with defaults if missing.
    """
    defaults = {
        "total_points": 1000000,
        "dimensions": 2,           
        "radius": 0.5,
        "convergence": 1.0e-6
    }
    
    if not os.path.exists(filename):
        print(f"Configuration file '{filename}' not found. Generating default...")
        with open(filename, "w") as f:
            for key, value in defaults.items():
                f.write(f"{key}={value}\n")
        return defaults
    
    config = {}
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                key, val = line.split("=")
                if "." in val:
                    config[key.strip()] = float(val.strip())
                else:
                    config[key.strip()] = int(val.strip())
    return config
    

def PI():
    """
    compute PI to be treated as reference for convergence calculation"""
    pi = math.pi
    return pi
    
def local_montecarlo(rng, local_points, dimensions):
    """
    The vectorized Monte Carlo engine.
    Generates random coordinates and counts how many fall inside the unit circle/sphere.
    """
    # CRITICAL: Independent random seeds based on rank to avoid duplicated points
    batch_size = 100000;
    points_inside = 0
    for i in range(0,local_points,batch_size):
        current_batch = min(batch_size,local_points - i)
    # Generate random points between -1.0 and 1.0 in a fully vectorized way
    # Shape will be (local_points, 2) for 2D or (local_points, 3) for 3D
        coordinates = rng.uniform(0.0, 1.0,(current_batch, dimensions))
    
    # Calculate the squared distance from the origin: x^2 + y^2 (+ z^2)
    # Using np.sum along axis 1 sums the row wise means  
        squared_distances = np.sum((coordinates- 0.5)**2, axis=1)
    
    # Count how many points lie inside circle
        points_inside += np.sum(squared_distances <= 0.25)
    
    return int(points_inside)

def main():
    # --- 1. MPI INITIALIZATION ---
    comm = MPI.COMM_WORLD
    my_rank = comm.Get_rank()
    nproc = comm.Get_size()
    dimensions = 2
    radius = 0.5
    config = None
    total_points =None

    # --- 2. INPUT HANDLING (Root Only) ---
    if my_rank == 0:
        print(f"--- Monte Carlo Simulation started on {nproc} MPI Processes ---")
        config = load_or_create_config(FILE_CONFIG)
        print(f"Parameters loaded: {config}")
        #total_points = int(config["total_points"])
        #convergence = config["convergence"]
        

    # --- 3. BROADCAST PARAMETERS ---
    # Root sends the config dictionary to all other processes
    config = comm.bcast(config, root=0)
    total_points = int(config["total_points"])
    dimensions = int(config["dimensions"])
    radius = float(config["radius"])
    convergence = float(config["convergence"])
    pi_estimate = 0
    Pi = PI()
    # create a random number generator object
    rng = np.random.default_rng(42+my_rank)
    # Start timer
    comm.Barrier()
    start_time = MPI.Wtime()
    while abs(Pi- pi_estimate)>convergence:
        
    # Calculate points for this specific process (handle remainders gracefully)
        local_points = int(total_points/nproc)
        remainder = total_points % nproc
        if my_rank < remainder:
            local_points = local_points + 1
        else:
            local_points 

    # --- 4. PARALLEL COMPUTATION ---
    # No Scatter needed. Every process just generates its own points natively.
        local_inside = local_montecarlo(rng, local_points, dimensions)

    # --- 5. REDUCE (Data Gathering) ---
    # We don't need a huge array back. We just need to SUM all the local_inside integers.
    # We must use numpy arrays for uppercase MPI commands.
        send_buf = np.array(local_inside, dtype=np.int64)
        recv_buf = np.array(0, dtype=np.int64)

    # All processes send their single integer, Root receives the mathematical SUM
        comm.Reduce(send_buf, recv_buf, op=MPI.SUM, root=0)

        
        

    # --- 6. MATH & OUTPUT HANDLING (Root Only) ---
        if my_rank == 0:
            global_inside = recv_buf.item()
        
        # Calculate Pi based on dimensions
            if dimensions == 2:
            # Circle: Area of square is 4. Area of circle is Pi. Ratio is Pi/4.
                pi_estimate = 4.0 * (global_inside / total_points)
                result_val = pi_estimate * (radius**2)
                metric_name = "Area"
                difference = abs(pi_estimate - Pi)
                print(f"The difference is : {difference}")
                if abs(Pi- pi_estimate)> convergence:
                    total_points = total_points + 50
                
            
            elif dimensions == 3:
            # Sphere: Volume of cube is 8. Volume of sphere is 4/3 Pi. Ratio is Pi/6.
                pi_estimate = 6.0 * (global_inside / total_points)
                result_val = (4.0 / 3.0) * pi_estimate * (radius**3)
                metric_name = "Volume"
                difference = abs(pi_estimate - Pi)
                print(f"The difference is : {difference}")
                if abs(Pi- pi_estimate)> convergence:
                    total_points = total_points + 50
                    

        pi_estimate = comm.bcast(pi_estimate, root=0)
        total_points = comm.bcast(total_points, root=0)
        
    comm.Barrier()
    end_time = MPI.Wtime()
    
    if my_rank ==0:
        
        execution_time = end_time - start_time
        # Print results
        print("\n--- SIMULATION RESULTS ---")
        print(f"Estimated Pi value : {pi_estimate:.14f}")
        print(f"Calculated {metric_name}  : {result_val:.8f}")
        print(f"MPI Execution Time : {execution_time:.4f} seconds")
        print(f"MPI total points that leads to convergence:{total_points}")

        #Save report to text file
        with open(FILE_OUTPUT, "w") as f:
            f.write("--- MONTE CARLO HPC REPORT ---\n")
            f.write(f"MPI Processes : {nproc}\n")
            f.write(f"Total Points  : {total_points}\n")
            f.write(f"Dimensions    : {dimensions}D\n")
            f.write(f"Radius        : {radius}\n")
            f.write(f"Pi Estimate   : {pi_estimate:.14f}\n")
            f.write(f"Final {metric_name}    : {result_val:.8f}\n")
            f.write(f"Compute Time  : {execution_time:.4f} seconds\n")
            
        print(f"Results saved to {FILE_OUTPUT}")

if __name__ == "__main__":
    main()


zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python mpi_sim_v2.py

--- SIMULATION RESULTS ---  
Estimated Pi value : 3.14159235393331  
Calculated Area  : 0.78539809  
MPI Execution Time : 131.9555 seconds  
MPI total points that leads to convergence:1156150  
Results saved to /home/zia__/jupyter/mc_results_mpi.txt  
(jlab-env) zia__@DESKTOP-OVKLL81:~/jupyter$ cat mc_results_mpi.txt  
--- MONTE CARLO HPC REPORT ---  
MPI Processes : 2  
Total Points  : 1156150  
Dimensions    : 2D  
Radius        : 0.5  
Pi Estimate   : 3.14159235  
Final Area    : 0.78539809  
Compute Time  : 131.9555 seconds  